# GlyphBench raw results — Qwen3.5-{4B, 9B}

Per-suite subplot grids of raw episodic returns for the two target models. Each subplot is one environment; both models are plotted side-by-side with their individual episodes as scatter points plus mean ± std bars.

In [ ]:
import json
import os
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Override via env var, e.g. `GLYPHBENCH_RESULTS_DIR=/path/to/runs/evals jupyter ...`
RESULTS_DIR = Path(os.environ.get('GLYPHBENCH_RESULTS_DIR', '../runs/evals'))
MODELS = ['Qwen/Qwen3.5-4B', 'Qwen/Qwen3.5-9B']
MODEL_COLORS = {'Qwen/Qwen3.5-4B': '#1f77b4', 'Qwen/Qwen3.5-9B': '#d62728'}
SUITE_ORDER = ['minigrid', 'minihack', 'atari', 'classics', 'craftax', 'procgen']
EXPECTED = {'minigrid': 71, 'minihack': 63, 'atari': 57, 'classics': 50, 'craftax': 35, 'procgen': 16}
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 110,
    'savefig.dpi': 200,
    'font.size': 8,
    'axes.titlesize': 7,
    'axes.labelsize': 8,
    'xtick.labelsize': 6,
    'ytick.labelsize': 6,
    'legend.fontsize': 9,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## Load all episodic returns into a flat DataFrame

In [ ]:
def load_episodes(results_dir: Path) -> pd.DataFrame:
    """Walk every results.jsonl under evals/<model>/<run-hash>/, one row per episode.

    Schema: model, env, suite, episodic_return, parse_failure_rate, episode_length, num_turns.
    """
    rows = []
    for model_dir in sorted(results_dir.iterdir()):
        model = model_dir.name.replace('glyphbench--Qwen--', 'Qwen/')
        if model not in MODELS:
            continue
        for jsonl in model_dir.rglob('results.jsonl'):
            with open(jsonl) as f:
                for line in f:
                    try:
                        r = json.loads(line)
                    except json.JSONDecodeError:
                        continue
                    info = r.get('info') or {}
                    env = info.get('env_id')
                    er = r.get('episodic_return')
                    if env is None or er is None or 'glyphbench/' not in env:
                        continue
                    suite = env.replace('glyphbench/', '').split('-')[0]
                    rows.append({
                        'model': model,
                        'env': env,
                        'suite': suite,
                        'episodic_return': er,
                        'parse_failure_rate': r.get('parse_failure_rate', 0.0),
                        'episode_length': r.get('episode_length', 0),
                        'num_turns': r.get('num_turns', 0),
                    })
    return pd.DataFrame(rows)

df = load_episodes(RESULTS_DIR)
print(f'Loaded {len(df):,} episodes')
print(f'Models: {sorted(df.model.unique())}')
print(f'Suites: {sorted(df.suite.unique())}')
print(f'Unique envs covered: {df.env.nunique()}')
df.head()

## Coverage summary (envs with ≥1 episode landed)

In [ ]:
summary = (
    df.groupby(['model', 'suite']).env.nunique().unstack(fill_value=0)
      .reindex(columns=SUITE_ORDER)
)
for s in SUITE_ORDER:
    summary[s] = summary[s].astype(str) + '/' + str(EXPECTED[s])
summary

## Per-suite raw-return subplot grids

One figure per suite. Each subplot is one env; x-axis is the two models, y-axis is episodic return. Individual episodes appear as jittered scatter points; mean ± std is overlaid as a horizontal bar with whiskers.

In [ ]:
def plot_suite(suite: str, df_suite: pd.DataFrame, ncols: int = 8) -> plt.Figure:
    envs = sorted(df_suite.env.unique())
    n = len(envs)
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 1.8, nrows * 1.7), squeeze=False)
    rng = np.random.default_rng(0)
    for idx, env in enumerate(envs):
        r, c = divmod(idx, ncols)
        ax = axes[r][c]
        for mi, model in enumerate(MODELS):
            sub = df_suite[(df_suite.env == env) & (df_suite.model == model)]
            if sub.empty:
                continue
            xs = mi + 0.07 * rng.standard_normal(len(sub))
            ax.scatter(xs, sub.episodic_return, s=14, alpha=0.55,
                       color=MODEL_COLORS[model], edgecolors='white', linewidths=0.4)
            mean = sub.episodic_return.mean()
            std = sub.episodic_return.std() if len(sub) > 1 else 0.0
            ax.errorbar(mi, mean, yerr=std, fmt='_', color=MODEL_COLORS[model],
                        elinewidth=1.2, capsize=4, markersize=14)
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['4B', '9B'])
        ax.set_xlim(-0.5, 1.5)
        ax.set_title(env.replace('glyphbench/', '').replace(f'{suite}-', ''), fontsize=6)
        ax.tick_params(axis='y', labelsize=6)
    for idx in range(n, nrows * ncols):
        r, c = divmod(idx, ncols)
        axes[r][c].axis('off')
    fig.suptitle(
        f'{suite.upper()} — raw episodic returns ({df_suite.env.nunique()}/{EXPECTED[suite]} envs covered)',
        fontsize=11, y=1.0,
    )
    fig.tight_layout()
    return fig

for suite in SUITE_ORDER:
    df_s = df[df.suite == suite]
    if df_s.empty:
        print(f'{suite}: no data')
        continue
    fig = plot_suite(suite, df_s)
    fig.savefig(FIGURES_DIR / f'raw_returns_{suite}.png', bbox_inches='tight')
    plt.show()
    plt.close(fig)

## Per-model overall mean per suite (one bar chart)

In [ ]:
agg = (
    df.groupby(['suite', 'model']).episodic_return
      .agg(['mean', 'std', 'count'])
      .reset_index()
)
fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(SUITE_ORDER))
width = 0.38
for mi, model in enumerate(MODELS):
    sub = agg[agg.model == model].set_index('suite').reindex(SUITE_ORDER)
    ax.bar(x + (mi - 0.5) * width, sub['mean'], width,
           yerr=sub['std'], label=model.replace('Qwen/', ''),
           color=MODEL_COLORS[model], capsize=3, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels([s.title() for s in SUITE_ORDER])
ax.set_ylabel('Mean episodic return ± std (across all envs in suite)')
ax.set_title('Per-suite mean episodic return — Qwen3.5-4B vs Qwen3.5-9B')
ax.axhline(0, color='gray', lw=0.5)
ax.legend(loc='upper right', frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'per_suite_mean.png', bbox_inches='tight')
plt.show()

## Per-env mean comparison (4B vs 9B scatter)

In [ ]:
per_env = (
    df.groupby(['env', 'suite', 'model']).episodic_return.mean().unstack('model')
)
per_env = per_env.dropna(subset=MODELS)
fig, ax = plt.subplots(figsize=(6.5, 6.5))
for suite in SUITE_ORDER:
    pts = per_env[per_env.index.get_level_values('suite') == suite]
    if pts.empty:
        continue
    ax.scatter(pts['Qwen/Qwen3.5-4B'], pts['Qwen/Qwen3.5-9B'],
               s=28, alpha=0.7, label=suite, edgecolors='white', linewidths=0.5)
lims = [min(per_env.min().min(), 0), max(per_env.max().max(), 1)]
ax.plot(lims, lims, 'k--', alpha=0.3, lw=1, label='y = x')
ax.set_xlabel('Qwen3.5-4B  mean episodic return')
ax.set_ylabel('Qwen3.5-9B  mean episodic return')
ax.set_title(f'Per-env comparison — {len(per_env)} envs covered by both models')
ax.legend(loc='best', fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(FIGURES_DIR / 'per_env_4b_vs_9b.png', bbox_inches='tight')
plt.show()